# Data Preprocessing for Train Data
This notebook contains standard data preprocessing steps for the Football match prediction dataset.

In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neural_network import MLPClassifier
#%pip install xgboost
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import accuracy_score, f1_score, log_loss, cohen_kappa_score, brier_score_loss

In [2]:
# load the features_train_with_player_position_v2.csv
df_train = pd.read_csv("../data/features_train_with_player_position_v2.csv")
df_train.head()
df_train.shape

(12303, 198)

In [3]:
#drop the id column
df_train.drop(columns=["ID"], inplace=True)   

#split the data into X and y
X_train = df_train.drop(columns=["target"])
y_train = df_train["target"]   

In [4]:
#split 80% of the data into training and 20% into testing  
X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.2, random_state=42)


In [5]:
#scale the data using StandardScaler
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [6]:
# Print class imbalance and introduce class weighting

# Print class distribution for y_train
y_counts = y_train.value_counts()
print('Class distribution in y_train:')
print(y_counts)

# Compute class weights for y_train
classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))
print('Class weights:', class_weight_dict)


Class distribution in y_train:
target
2.0    4273
0.0    3003
1.0    2566
Name: count, dtype: int64
Class weights: {0.0: 1.0924630924630925, 1.0: 1.2785138997142116, 2.0: 0.7677665964583821}


In [28]:
# Train and evaluate 5 models using Stratified K-Fold Cross-Validation
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(class_weight=class_weight_dict, max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(class_weight=class_weight_dict, random_state=42),
    'XGBoost': XGBClassifier(scale_pos_weight=class_weight_dict.get(1, 1), use_label_encoder=False, eval_metric='logloss', random_state=42),
    'Neural Network': MLPClassifier(max_iter=500, random_state=42)
}

metrics_cv = {name: [] for name in models.keys()}
metrics_cv['Soft Voting Ensemble'] = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\nFold {fold+1}")
    X_tr, X_te = X_train.iloc[train_idx], X_train.iloc[test_idx]
    y_tr, y_te = y_train.iloc[train_idx], y_train.iloc[test_idx]

    # Standardize features
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_te_scaled = scaler.transform(X_te)

    fold_models = {}
    for name, model in models.items():
        mdl = model.__class__(**model.get_params())
        mdl.fit(X_tr_scaled, y_tr)
        y_pred = mdl.predict(X_te_scaled)
        y_proba = mdl.predict_proba(X_te_scaled)
        fold_models[name] = mdl
        metrics_cv[name].append([
            accuracy_score(y_te, y_pred),
            f1_score(y_te, y_pred, average='macro'),
            log_loss(y_te, y_proba),
            cohen_kappa_score(y_te, y_pred)
        ])

    # Soft Voting Ensemble
    ensemble = VotingClassifier(
        estimators=[
            ('lr', fold_models['Logistic Regression']),
            ('rf', fold_models['Random Forest']),
            ('xgb', fold_models['XGBoost']),
            ('mlp', fold_models['Neural Network'])
        ],
        voting='soft'
    )
    ensemble.fit(X_tr_scaled, y_tr)
    y_pred_ensemble = ensemble.predict(X_te_scaled)
    y_proba_ensemble = ensemble.predict_proba(X_te_scaled)
    metrics_cv['Soft Voting Ensemble'].append([
        accuracy_score(y_te, y_pred_ensemble),
        f1_score(y_te, y_pred_ensemble, average='macro'),
        log_loss(y_te, y_proba_ensemble),
        cohen_kappa_score(y_te, y_pred_ensemble)
    ])

# Aggregate metrics
import numpy as np
metrics_mean = {name: np.mean(metrics_cv[name], axis=0) for name in metrics_cv}
metrics_df = pd.DataFrame.from_dict(
    metrics_mean,
    orient='index',
    columns=['Accuracy', 'Macro-F1', 'Log-loss', 'Cohen Kappa']
)
print("\nMean metrics across folds:")
print(metrics_df)



Fold 1

Fold 2

Fold 3

Fold 4

Fold 5

Fold 6

Fold 7

Fold 8

Fold 9

Fold 10

Mean metrics across folds:
                      Accuracy  Macro-F1  Log-loss  Cohen Kappa
Logistic Regression   0.435581  0.415968  1.076104     0.138691
Random Forest         0.461793  0.356071  1.049467     0.112354
XGBoost               0.442794  0.384254  1.135179     0.109008
Neural Network        0.385388  0.367129  5.195032     0.055051
Soft Voting Ensemble  0.422779  0.389825  1.086951     0.097751
